# AO Data analysis

In [ ]:
import numpy as np
import aotools
import matplotlib.pyplot as plt
import astropy.io.fits as pfits
from IPython.display import HTML
from matplotlib import animation

## 1. About the Telemetry data

_This workshop will use telemetry datasets acquired on the AO bench from Institut d'Optique (Palaiseau, France)._

The Shack-Hartman wavefront sensor has an 8×8 microlenses array, with the shape of the pupil illumination, 3 lenslets are unused at each corner, which totals 52 sub-apertures. For each subaperture, we obtain local slope along x and y axis.

The deformable mirror has 9×9 actuators (Fried Geometry with also 12 unused actuators, for a total of 69 actuators.

A standard telemetry records contains at least two time sequences:
- WFS measurements for each iteration
- DM commands for each iteration

Aditionnaly, the interaction matrix is required to transform DM commands into the equivalent WFS measurements, and the sample frequency is needed for time analysis (here, __Fs = 100 Hz__).
![actus_and_subaps.png](actus_and_subaps.png)

In [ ]:
# Loading data from a close loop run
Fs = 100
y_cl = pfits.getdata("CL_slopes.fits")
u_cl = pfits.getdata("CL_commands.fits")
time = np.arange(y_cl.shape[1]) * 1/Fs

## 2. Waverfront Sensor measurement and DM commands

Let's look at the shape of the measurement array

In [ ]:
print(y_cl.shape)

The first dimension is the number of measurement, so twice the number of SH sub apertures, and the data is organized with all x measurement first (from 0 to 51 included) and all y measurement then (from 52 to the end).
The second dimension is time.

Measurement are given as the local wavefront slope in radians.

We can look at the measurement vector for one iteration :

In [ ]:
plt.plot(y_cl[:, 123])
plt.xlabel("measurement number")

and to the time trace of one given measurement :

In [ ]:
plt.plot(y_cl[72, :])
plt.xlabel("time (iteration number)")

An interesting visualisation is to superimpose all timetraces with partially transparent curves. This will vlearly show the range of values, as well as eventual outsiders.

We can also separate x and y slopes into two subplots.

In [ ]:
fig, [ax1, ax2] = plt.subplots(figsize=(10, 4), ncols=2, sharey=True)
for i in range(52):
    ax1.plot(time, 1e3*y_cl[i, :], color="#ff990022") #in mrad
    ax2.plot(time, 1e3*y_cl[i+52, :], color="#ff009922") #in mrad
ax1.set_ylabel("measured slope (mrad)")
ax1.set_xlabel("time (s)"); ax2.set_xlabel("time (s)")
ax1.set_title("x slopes"); ax2.set_title("y slopes")
fig.suptitle("Closed loop measurements", color='g')

The first iterations where the measured values are all higher is called the bootstrap time. It is the time necessary to reach stable performance after closing the loop.
This time is usually not included in the data analysis.
We can zoom in by looking at data until iteration 80.

In [ ]:
fig, [ax1, ax2] = plt.subplots(figsize=(10, 4), ncols=2, sharey=True)
for i in range(52):
    ax1.plot(1e3*y_cl[i, :80], color="#ff990022") #in mrad
    ax2.plot(1e3*y_cl[i+52, :80], color="#ff009922") #in mrad
ax1.set_ylabel("measured slope (mrad)")
ax1.set_xlabel("time (s)"); ax2.set_xlabel("time (iteration number)")
ax1.set_title("x slopes"); ax2.set_title("y slopes")
fig.suptitle("Closed loop measurements, zoom in bootstrap")

The command array has the number of DM actuators as the first dimension, and time as the second dimension.

In [ ]:
for i in range(69):
    plt.plot(time, u_cl[i, :], color="#00779922")
plt.ylabel("DM command (µm)")
plt.xlabel("time (s)")

## 3. Pseudo open-loop measurements

To analyse performance, it is interesting to compare the closed loop system with an open loop system. However, it is usually not a good idea to take data without any correction:
- depending on the perturbation, the measurement might be out of the linear range of the wavefront sensor;
- the conditions may evolve between the two acquisition, and the comparison might then not be relevant.

We need a way to estimate measurement in open loop from our telemetry data in close loop.
In terms of phase propagation, the DM dephasing is added to the incoming wavefront, which gives the residual wavefront.
This residual wavefront is then measured by the WFS.
It then comes that removing the commands to the measured residual wavefront gives an estimation of the measures that would be obtained without AO correction. However, this operation requires two things:
- a knowledge of how each command affects the measured slopes → given by the interaction matrix
- an estimation of the loop delay to understand which command in the time series affects the slopes at a given time.

In our case, the complete loop delay is 1 frame.

In [ ]:
imat = pfits.getdata("imat.fits").T

Let's observe the interaction matrix:

In [ ]:
plt.imshow(imat)
plt.colorbar()

In [ ]:
actu_index = 12 # <69
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(imat[:52, actu_index], color="#ff9900")
plt.subplot(1, 2, 2,)
plt.plot(imat[52:, actu_index], color="#ff0099")

This display gives information about the system geometry and data organisation:
- two separate blocks confirms that the slopes are organized with all for one axis, then all for the other axis.
- the slopes index show there is an inversion (optical focus) between the DM and WFS that is not taken into account : the first actuator is mostly visible by the last slope in the list.
- the direction of DM actuators ordering is 90° turned with respect of the WFS reading. ??? TBC
(Compare to the interaction matrix from simulation workshop)

We can then compute pseudo open-loop measurement and look at time traces for every subapertures:

In [ ]:
u_in_y_space = imat @ u_cl #project commands in WFS measurement space
y_pol = y_cl[:, 1:] - u_in_y_space[:, :-1] #index shifts account for 1 frame delay

fig, [ax1, ax2] = plt.subplots(figsize=(10, 4), ncols=2, sharey=True)
for i in range(52):
    ax1.plot(time[1:], 1e3*y_pol[i, :], color="#ff990022") #in mrad
    ax2.plot(time[1:], 1e3*y_pol[i+52, :], color="#ff009922") #in mrad
ax1.set_ylabel("measured slope (mrad)")
ax1.set_xlabel("time (s)"); ax2.set_xlabel("time (s)")
ax1.set_title("x slopes"); ax2.set_title("y slopes")
fig.suptitle("Pseudo open loop measurements", color='r')

In [ ]:
print(y_pol.shape)
print(y_cl.shape)

Lets crop the data to remove the bootstrap iterations form the close loop data, and also crop the POL data to get equal size arrays.

In [ ]:
y_pol_c = y_pol[:, 19:]
y_cl_c = y_cl[:, 20:]
time_c = time[20:]
print(y_pol_c.shape)
print(y_cl_c.shape)
print(time_c.shape)

## 4. A first performance index

With a simple integrator, the main goal of the regulator is to drive the closed loop measurement to zero.
A time variance (or standard deviation) can give information about where in the pupil the residual phase is moving a lot.

In [ ]:
fig, (ax1, ax2) = plt.subplots(figsize=(10, 4), ncols=2, sharey=True)
ax1.plot(1e3*np.std(y_pol_c[:52, :], axis=1), label="x slopes", color="#ff9900")
ax1.plot(1e3*np.std(y_pol_c[52:, :], axis=1), label="y slopes", color="#ff0099")
ax1.set_title("Open loop", color='r')
ax1.set_ylabel("slope temporal std (mrad of dev)")
ax1.set_xlabel("subaperture number")
ax1.grid()
ax2.plot(1e3*np.std(y_cl_c[:52, :], axis=1), label="x slopes", color="#ff9900")
ax2.plot(1e3*np.std(y_cl_c[52:, :], axis=1), label="y slopes", color="#ff0099")
ax2.set_title("Close loop", color='g')
ax2.set_xlabel("subaperture number")
ax2.grid()
plt.legend()

## 5. Angle of arrival

With a SH-WFS delivering x and y WF slopes for each subaperture, it is possible to estimate a global tip (or tilt) by averaging x (or y) measurement on all subapertures for each iteration.

With atmospheric turbulence, tip and tilt are usually the most energetic modes of perturbation. Looking at their evolution in closed loop, and comparing with open loop gives a good insight about how our loop is performing on the low orders modes.

In [ ]:
aoa_x_pol = np.mean(y_pol_c[:52, :], axis=0)
aoa_y_pol = np.mean(y_pol_c[52:, :], axis=0)
aoa_x_cl = np.mean(y_cl_c[:52, :], axis=0)
aoa_y_cl = np.mean(y_cl_c[52:, :], axis=0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(figsize=(10, 4), ncols=2, sharey=True)
ax1.plot(time_c, 1e3*aoa_x_pol, label="x slopes", color="#ff9900")
ax1.plot(time_c, 1e3*aoa_y_pol, label="y slopes", color="#ff0099")
ax1.set_title("Open loop", color='r')
ax1.set_ylabel("Global angle of arrival (mrad)")
ax1.set_xlabel("subaperture number")
ax1.grid()
ax2.plot(time_c, 1e3*aoa_x_cl, label="x slopes", color="#ff9900")
ax2.plot(time_c, 1e3*aoa_y_cl, label="y slopes", color="#ff0099")
ax2.set_title("Close loop", color='g')
ax2.set_xlabel("subaperture number")
ax2.grid()
plt.legend()

## 6. Phase reconstruction

When a calibration matrix is performed with a modal basis (for example Zernike modes), it is possible to derive a reconstructor matrix that allows to project the measurement onto the Zernike space.

In [ ]:
reconstructor = pfits.getdata("Rmap.fits")

The projection is a matric-vector multiplication. In our case, since time is the second dimension, we can perform a matrix-matrix multiplication that will create an array with time as the second coordinate and modal decomposition as the first coordinate.

In [ ]:
phiz_pol = reconstructor @ y_pol_c
phiz_cl = reconstructor @ y_cl_c
print(phiz_pol.shape)

And once again look at the time trace of given modes:

In [ ]:
plt.plot(phiz_pol[0, :], label='tip', color="#ff9900")
plt.plot(phiz_pol[1, :], label='tilt', color="#ff0099")
#plt.plot(phiz_pol[6, :], label='coma')
plt.legend(); plt.ylabel("Mode coefficient"); plt.xlabel("time")

In [ ]:
plt.plot(phiz_cl[0, :], label='tip', color="#ff9900")
plt.plot(phiz_cl[1, :], label='tilt', color="#ff0099")
#plt.plot(phiz_cl[6, :], label='coma')
plt.legend(); plt.ylabel("Mode coefficient"); plt.xlabel("time")

This series of Zernike modes coefficients can be used to reconstruct the shape of the wavefront, within the measurement capabilities of the WFS. As a reminder, we use a function from `aotools` to compute a series of zernike modes phase screens, as some of the are shown below.

In [ ]:
Zs = aotools.zernikeArray(105, 128)[1:, :, :]
for i in range(28):
    plt.subplot(4, 7, i+1)
    plt.imshow(Zs[i, :, :])
    plt.axis('off')

By computing a weighted sum of these phase screens, using zernike coefficients as weights, we can obtain an estimated phase screen corresponding to each measurement vector.

In [ ]:
# Pseudo open loop phase
w = phiz_pol[:, 35]
phizer = np.sum(Zs * w[:, None, None], axis=0)
plt.imshow(phizer.T, origin='lower')
plt.colorbar()

In [ ]:
w = phiz_cl[:, 35]
phizer = np.sum(Zs * w[:, None, None], axis=0)
plt.imshow(phizer.T, origin='lower')
plt.colorbar()

## 7. Phase variance
From the series of zernike coefficients, and provided that the reconstructor is properly calibrated, we can obtain an estimation of the phase __spatial__ variance by looking at the sum of the coefficients squared.

In [ ]:
plt.plot(time_c, np.sum(phiz_pol**2, axis=0), 'r', label="pseudo open loop")
plt.plot(time_c, np.sum(phiz_cl**2, axis=0), 'g', label="close loop")
plt.ylabel("phase variance (rad²) across the pupil")
plt.xlabel("time (s)")
plt.legend()

Following the Mahajan approximation, the residual phase variance can then be used to estimate the Strehl ratio... in good conditions!

In [ ]:
srse = np.exp(-np.sum(phiz_cl**2, axis=0))
plt.plot(time_c, srse, 'g')
plt.xlabel("time (s)"); plt.ylabel("Strehl Ratio (%)")
plt.title("Strehl ratio estimation in close loop")

In [ ]:
plt.plot(time_c, srse, 'g', alpha=0.3, label="short exposure")
srle = np.cumsum(srse)/np.arange(1, 1181)
plt.plot(time_c, srle, 'g', label="long exposure")
plt.xlabel("time (s)"); plt.ylabel("Strehl Ratio (%)")
plt.grid(); plt.legend()

From the pseudo open-loop measurement, we cans also look at the time variance for each mode. This will show how much each of these modes contribute to the total phase variance.

In [ ]:
modal_var = np.var(phiz_pol, axis=1)
noll = np.arange(1, 105)
plt.loglog(noll, modal_var)
modal_var = np.var(phiz_cl, axis=1)
plt.loglog(noll, modal_var)
plt.loglog(noll, 10*noll**(-11/6), '--') 
plt.grid(True, which='both')
plt.xlabel("Zernike mode number")
plt.ylabel("Temporal variance per mode")

We clearly see that the modes of the lowest order (thus containing the smallest spatial frequencies) are the most energetic ones. The log-log plot shows a linear trend (underlined by the orange dash line) which correspond to a the power law described in the Kolmogorov and Von Karman theories of the atmospheric turbulence.
The deviation for high order modes can be explained by the noise on the wavefront sensor, the worse reconstruction for these modes or imperfection iin the phase screen manufacturing.

## 7. Frequency domain study
In many systems where the atmospheric turbulence is not the only source of perturbation, it can be intersting to look at the frequency behaviour of the measurement

In [ ]:
from scipy import signal
mode_index = 1
f, Pxx_den = signal.welch(phiz_pol[mode_index, :], 100, nperseg=500)
plt.semilogy(f, Pxx_den, 'r', label='pseudo open loop')
f, Pxx_den = signal.welch(phiz_cl[mode_index, :], 100, nperseg=500)
plt.semilogy(f, Pxx_den, 'g', label='close loop')
plt.xlabel("frequency (Hz)")
plt.ylabel("Power spectral density")
plt.title(f"mode number {mode_index}")
plt.legend()